# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook guides you through loading and exploring the FAIR^2 Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors dataset using the `mlcroissant` library.

### Dataset Source
The dataset schema is accessible via the following Croissant schema URL and fully utilizes unique `@id` references for dataset assets.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their constituent fields and columns using unique `@id` references.

Below, we enumerate all record sets in the dataset, and for each record set, its fields and their column mappings. All entities are reported by their `@id` fields.

In [ ]:
# List all record sets by @id and enumerate their fields
record_set_ids = []

print("Available record sets:\n----------------------")
for record_set in dataset.record_sets:
    print(f"@id: {record_set['@id']}")
    record_set_ids.append(record_set['@id'])
    
    if 'field' in record_set and record_set['field']:
        print("  Fields:")
        fields = record_set['field']
        # If only one field, make it a list
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            field_id = field['@id'] if isinstance(field, dict) and '@id' in field else str(field)
            print(f"    - @id: {field_id}")
    else:
        print("  No fields defined.")
    print("")

# For demonstration, show a brief sample of records from the first record set (if any).
if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print(f"Sample records from record set @id: {example_record_set_id}")
    for i, record in enumerate(dataset.records(record_set=example_record_set_id)):
        print(record)
        if i >= 2:
            print("...\n(Only showing first 3 records)")
            break
else:
    print("No record sets found in this dataset.")

## 3. Data Extraction
Load data from a specific record set (referenced by its `@id`) into a DataFrame for analysis. Use the record set `@id` and fields discovered above.

We'll demonstrate loading each record set individually.

In [ ]:
# Extract all record sets into DataFrames using their @id
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set '@id': {record_set_id}")
    print(f"Columns (field @id): {df.columns.tolist()}")
    print(df.head(2)) # Only show the first 2 records for preview
    print("\n---\n")

# For demonstration, pick the first record set
if record_set_ids:
    chosen_record_set_id = record_set_ids[0]
    print(f"Selected record set for further analysis: {chosen_record_set_id}")
    print(f"Fields in this set: {dataframes[chosen_record_set_id].columns.tolist()}")
    display(dataframes[chosen_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
We will now perform sample EDA by:
- Filtering numeric fields,
- Normalizing values,
- Grouping by a categorical field.

All columns/fields are referenced by their `@id`.

In [ ]:
# Choose the record set for analysis
df = dataframes[chosen_record_set_id].copy()
print(f"Working with record set @id: {chosen_record_set_id}")

# Identify a numeric field (by @id, as required)
# For demonstration, we will automatically pick the first numeric-looking column
numeric_field_id = None
for col in df.columns:
    try:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
        # Try to convert to numeric for checking
        pd.to_numeric(df[col])
        numeric_field_id = col
        df[col] = pd.to_numeric(df[col], errors='coerce')
        break
    except:
        continue

if numeric_field_id is None:
    print("No numeric field found for EDA.")
else:
    print(f"Using numeric field @id: {numeric_field_id}")

    # Set a threshold for demonstration
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 10

    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold} (Count: {len(filtered_df)}):")
    print(filtered_df.head())

    # Add normalized column
    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, normalized_col]].head())

    # Identify a group/categorical field for grouping (not the numeric field)
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id and df[col].dtype == object:
            group_field_id = col
            break

    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of {numeric_field_id} by field @id: {group_field_id}")
        print(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")

## 5. Visualization
Visualize the distribution of the selected numeric field and, if available, its average grouped by the selected categorical field. All axes are labeled by their respective `@id` values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and df[numeric_field_id].notnull().any():
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.title(f'Distribution of Numeric Field (@id: {numeric_field_id})')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Grouped bar plot if grouping field exists
    if group_field_id is not None:
        # Aggregate mean per group for barplot demonstration
        plt.figure(figsize=(10,4))
        group_means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
        group_means.plot(kind='bar')
        plt.title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field for visualization.")

## 6. Conclusion

- This notebook demonstrated how to programmatically access, explore, and analyze a FAIR dataset described with the Croissant schema and referenced by `@id` using the `mlcroissant` library.
- All data structure references (record sets, fields, columns) rely exclusively on their `@id` values, ensuring robust, schema-consistent exploration.
- Further data analysis can be performed by leveraging the auto-extracted dataframes and unique Croissant schema identifiers.

For more complex pipelines or further documentation, see https://github.com/mlcommons/croissant.